# CausalJEPA — Rung 3: the learned eye (Discrete-JEPA tokenizer on pixels)

Swaps OCAtari's free ground-truth state for a **learned** Discrete-JEPA tokenizer; everything
else (Gumbel-Max abduction, seed-replay oracle, err_CF-vs-err_IV) stays structurally identical
to Rungs 1–2.5. The question (Problem B): does a predict-only eye keep the noise signal
(stuck-vs-not paddle motion) that counterfactuals need?

**Stage order: 0 → 1 → 2 (GATE) → 3 → 4.** Never run Stage 3 before the Stage-2 gate passes.

**Colab contract**: Drive is mounted first; every artifact lives under
`/content/drive/MyDrive/cjepa_rung3/`; every stage resumes from its last checkpoint —
a disconnect costs at most one checkpoint interval. Stages 0/2/4 are CPU-fine;
only Stage 1 (and lightly 3) wants the GPU. Run Stage 0 on a CPU runtime once,
then switch to GPU for Stage 1.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# --- get the code (set your repo URL; or keep a copy of the repo on Drive) ---
REPO_URL = 'https://github.com/<YOU>/counterfactual-world-model.git'  # <-- EDIT
import os, sys
if not os.path.exists('/content/counterfactual-world-model'):
    !git clone $REPO_URL /content/counterfactual-world-model
%cd /content/counterfactual-world-model
!git pull
sys.path.insert(0, '/content/counterfactual-world-model')

!pip -q install ale-py ocatari 'numpy>=2' --upgrade-strategy only-if-needed
# torch / sklearn / PIL / matplotlib ship with Colab
import ocatari, ale_py, torch
print('cuda:', torch.cuda.is_available())


## Stage 0 — collect WITH pixels + the pixel oracle (CPU runtime, run ONCE)
Rung-2-identical rollouts (ALE sticky `s∈{0,0.5}`, fs=2, same seeds) + 84×84 frames;
then the seed-replay oracle caches the CF / factual / other-branch next frames per eval
tuple. Both are resume-safe (per-episode shards; partial oracle cache every 25 tuples).
Both print the replay-determinism gate (must be 100% / 100%) and fire-rate ≈ s.


In [ ]:
!python -m pong_counterfactual.cjepa_rung3.stage0_collect
!python -m pong_counterfactual.cjepa_rung3.stage0_oracle


## Stage 1 — pretrain the eye (GPU runtime; the only heavy step)
Discrete-JEPA: I-JEPA-style context/EMA-target encoders + S2P/P2P latent prediction
+ VQ codebook (M=4 tokens/frame, K=64) — **no reconstruction loss**. Checkpoints to
Drive every 500 steps; re-running this cell after a disconnect RESUMES automatically.
Watch `perplexity` in the log — if it pins near 1 the codebook collapsed (turn knobs).


In [ ]:
!python -m pong_counterfactual.cjepa_rung3.stage1_pretrain --name M4_K64_res84


## Stage 2 — the GATE (minutes, CPU-fine). THE SOUL OF THIS RUNG
1. **Collision probe**: factual vs would-have-stuck next frames through the frozen eye —
   fraction mapping to identical tokens (Rung 2.5's orange curve, now on the real eye).
2. **Linear probe**: tokens → positions (resized px); player_y MAE must beat the paddle
   step (2.41 resized px).

**PASS = collision ≤ 25% (the Rung-2.5 knee) AND probe ok.** FAIL → retrain Stage 1 with
the knobs in order: `--K 128` → `--M 8` → loss weights → resolution ① (config.py `res=128`,
recollect). Every attempt is appended to `gate_log.json` — that table IS the Problem-B
evidence. **Train NO transition model until this passes.**


In [ ]:
!python -m pong_counterfactual.cjepa_rung3.stage2_gate --name M4_K64_res84


In [ ]:
# optional knob sweep when the gate fails (each config gets its own dir + gate row):
# !python -m pong_counterfactual.cjepa_rung3.stage1_pretrain --M 4 --K 128
# !python -m pong_counterfactual.cjepa_rung3.stage2_gate    --name M4_K128_res84
# !python -m pong_counterfactual.cjepa_rung3.stage1_pretrain --M 8 --K 128
# !python -m pong_counterfactual.cjepa_rung3.stage2_gate    --name M8_K128_res84
import json, pathlib
log = pathlib.Path('/content/drive/MyDrive/cjepa_rung3/gate_log.json')
if log.exists():
    for r in json.loads(log.read_text()):
        print(r['name'], 'collision', round(r['collision_allM'], 3),
              'py_MAE', round(r['probe_mae_resized_px']['player_y'], 2),
              'PASSED', r['PASSED'])


## Stage 3 — the forecaster on learned tokens (light; refuses if the gate failed)
`p(next frame's M tokens | last 8 frames' tokens + intended actions)` — M independent
K-way categoricals (decision ⑤), one model per s level. Resume-safe.


In [ ]:
!python -m pong_counterfactual.cjepa_rung3.stage3_transition --name M4_K64_res84


## Stage 4 — abduction vs intervention vs the pixel oracle (the headline)
`gumbel_abduction.py` unchanged, per token slot. Dual scoring (⑥): token-space primary,
probe-px secondary. Prints checks A–E + collision-split + copy baseline; writes
`results_rung3_<name>.json` and the one-plot headline figure to Drive.


In [ ]:
!python -m pong_counterfactual.cjepa_rung3.stage4_eval --name M4_K64_res84
from IPython.display import Image
Image('/content/drive/MyDrive/cjepa_rung3/rung3_headline_M4_K64_res84.png')


## Verdict template (the deliverable's paragraph 3)
Either: *“JEPA-style abstraction and counterfactuals coexist: at M=_, K=_, res=_ the eye
keeps the signal (collision _%), the headline holds (tok gap _, px gap _), and the
advantage concentrates on genuine stuck steps (stuck gap _ vs free _), not on collided
copy-steps.”* — or: *“at config _ the eye discards the signal (collision _%), the CF
degenerates to factual-copying (copy baseline wins / B fails) — the layer at fault is _
(gate_log.json rows _).”* Both are publishable; the diagnostics make the negative
result a result.
